# Attention language model

Start from the same Tiny Shakespeare token stream as the bigram model, then inspect token embeddings before adding attention.

In [ ]:
import random
import sys
from pathlib import Path

import torch
from torch import nn

repo_root = Path.cwd()
if not (repo_root / "data").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from src.dataset import get_batch, load_tiny_shakespeare_tokens, split_token_stream

## Prepare token streams and batches

In [ ]:
tokens, vocab, merges = load_tiny_shakespeare_tokens(repo_root / "data")
train_tokens, validation_tokens = split_token_stream(tokens)

vocab_size = len(vocab)
block_size = 8
batch_size = 32
n_embd = 32
head_size = 16

random.seed(42)
x_batch, y_batch = get_batch(
    "train", train_tokens, validation_tokens, block_size, batch_size
)
x_batch = torch.tensor(x_batch, dtype=torch.long)
y_batch = torch.tensor(y_batch, dtype=torch.long)

## Model scaffold

In [ ]:
class AttentionLanguageModel(nn.Module):
    def __init__(self, vocab_size, n_embd):
        super().__init__()
        self.head = self.Head(n_embd, head_size) # (B, T, H)
        self.token_embedding_table = self.token_embedding_table(vocab_size, n_embd)
        self.lm_head = nn.Linear(head_size, vocab_size)

    def forward(self, idx):
        x = self.token_embedding_table(idx)  # (B, T, C)
        x = self.head(x)  # (B, T, H)
        return self.lm_head(x) # (B, T, V)

    # # x: [B, T, C]
    # q,k,v: [B, T, H]
    # out: [B, T, H]
    class Head(nn.Module):
        def __init__(self, n_embd, head_size):
            super().__init__()
            self.n_embd = n_embd
            self.head_size = head_size
            self.query = nn.Linear(n_embd, head_size)
            self.key = nn.Linear(n_embd, head_size)
            self.value = nn.Linear(n_embd, head_size)

        def forward(self, x):
            B, T, C = x.shape   
            q = self.query(x)  # (B, T, H)
            k = self.key(x)  # (B, T, H)
            v = self.value(x)  # (B, T, H)

            positions = torch.arange(T, device=x.device)
            positional_embeddings = self.positional_embedding(positions)  # (T, H)

            attention = token_embeddings + positional_embeddings  # (B, T, C)
            attention = self.attention(q, k, v)  # (B, T, H)
            return attention

        def attention(self, q, k, v):
            # Compute attention scores
            attn_scores = torch.matmul(q, k.transpose(-2, -1)) / (self.head_size ** 0.5)  # (B, T, T)
            # Apply masking to attention scores
            T = attn_scores.shape[-1]
            mask = torch.triu(torch.ones(T, T), diagonal=1).bool()  # (T, T)
            attn_scores = attn_scores.masked_fill(mask, -float('inf'))
            # Normalize attention scores
            attn_weights = torch.softmax(attn_scores, dim=-1)  # (B, T, T)
            # Compute attention outputs
            out = torch.matmul(attn_weights, v)  # (B, T, H)
            return out


model = AttentionLanguageModel(vocab_size, n_embd)
head = model.Head(n_embd, head_size)
token_embeddings = model.token_embedding_table(x_batch)
print(f"Token embedding shape (B, T, C): {tuple(token_embeddings.shape)}")
attention = head(token_embeddings)
print(f"Attention shape (B, T, H): {tuple(attention.shape)}")

In [ ]:
#refactored into the above

# # Positional embeddings
# positional_embeddings = nn.Parameter(torch.zeros(1, block_size, n_embd))
# # Add positional embeddings to token embeddings
# token_embeddings += positional_embeddings
# print(f"Token embedding shape after adding positional embeddings (B, T, C): {tuple(token_embeddings.shape)}")

In [ ]:
# Causal self-attention

# # legacy cell, refactored into the head class above
# import math
# import torch
# from torch import nn

# key = nn.Linear(n_embd, head_size, bias=False)
# query = nn.Linear(n_embd, head_size, bias=False)
# value = nn.Linear(n_embd, head_size, bias=False)

# # x       [B, T, C]
# # q,k,v   [B, T, H]
# # scores  [B, T, T]
# # weights [B, T, T]
# # out     [B, T, H]
# # MATCH THE SHAPES OF THE LINEAR LAYERS!! 
# def causal_self_attention(x, mask=None):
#     k = key(x)
#     q = query(x)
#     v = value(x)

#     scores = q @ k.transpose(-1, -2) / math.sqrt(head_size)
#     scores = scores.masked_fill(mask == 0, float('-inf'))
#     weights = torch.softmax(scores, dim=-1)
#     attention_out = weights @ v
#     return attention_out

# x = torch.randn(1, 8, n_embd)
# mask = torch.ones(1, 8, 8)
# causal_self_attention(x, mask)

# # multiple heads


# TODO:

```
single causal head
→ multi-head attention
→ positional embeddings integrated into the model
→ train on Tiny Shakespeare
→ record validation loss
→ generate samples
→ compare against bigram
```